# Domain size, step by step and PC1–PC4 vs hold time

**Part 1** builds the two domain-size estimators in
`quantum_sim.utils.domains` one step at a time on a real shot. Every
intermediate array is plotted.

* **Approach A: first moment of the structure factor** (Fourier space)
  $L_A = 2\pi\,\langle S\rangle / \langle k\,S\rangle$ (`domain_length_px`)
* **Approach B: first zero of the pair correlation** (real space)
  $L_B$ = first $r$ where $g(r)=0$ (`radial_autocorr` + `first_zero`)

Each hand-written version is checked against the library function at the end
of its section, so this notebook cannot drift away from the code the analysis
scripts actually run.

**Part 2** animates the first four principal components of each hold time's
own PCA for the 700p (2026-08-19) and the 2026-09-11 datasets.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib import animation
from matplotlib.patches import Circle, Rectangle

from quantum_sim.config import dataset
from quantum_sim.io import ShotFile
from quantum_sim.utils.domains import domain_length_px, first_zero, radial_autocorr, window_2d
from quantum_sim.utils.images import centroid_crop_box, rotate_and_crop
from quantum_sim.utils.pca import align_signs, pca_svd, remove_components
from quantum_sim.utils.shots import hold_time_ms

%matplotlib inline
plt.rcParams["animation.embed_limit"] = 60  # MB; the jshtml movies below are a few MB each

## Loading

Both runs are framed the same way: a `crop_half` square about the centroid of
the t = 0 mean OD, then `rotate_and_crop` (45° rotation, inscribed square).
Aborted shots that have no `/images` group are skipped, and the dataset's
`max_time_ms` cut from `config/experiments.yaml` is applied. That cut is
800 ms for the 700p set and 2000 ms for 09-11.

In [ ]:
def load_dataset(name):
    # Rotated, cropped OD stack [shot, y, x] plus per-shot hold times [ms].
    ds = dataset(name)
    paths, t_ms = [], []
    for p in ds.shot_paths():
        with h5py.File(p, "r") as f:
            if "images" not in f:            # aborted shot, no frames
                continue
        t = hold_time_ms(ShotFile(p, cam=ds.camera), ds.time_index)
        if ds.max_time_ms is not None and t > ds.max_time_ms:
            continue
        paths.append(p)
        t_ms.append(t)
    t_ms = np.array(t_ms, dtype=float)

    t0_frames = [ShotFile(p, cam=ds.camera).od for p, t in zip(paths, t_ms) if t == 0]
    crop_box = centroid_crop_box(t0_frames, ds.crop_half)

    # ShotFile objects are not kept: each caches its frames, ~1.5 GB for 700 shots.
    stack = np.stack([rotate_and_crop(ShotFile(p, cam=ds.camera, crop_box=crop_box).od,
                                      ds.rotation_deg) for p in paths])
    times = np.unique(t_ms)
    print(f"{name}: {len(paths)} shots, {len(times)} hold times, stack {stack.shape}, "
          f"crop_box {crop_box}")
    return dict(ds=ds, paths=paths, t=t_ms, times=times, stack=stack, crop_box=crop_box)


data = {name: load_dataset(name) for name in ("shots700p", "sep11")}

# Part 1: the two domain-size estimators, step by step

### Step 0: the field both estimators act on

The demo is one 2026-09-11 shot at the hold time nearest `DEMO_T`. Before
either estimator runs, the field is prepared the same way as in the analysis
notebooks:

1. Raw OD, full camera frame. The red square is the centroid crop.
2. Crop, rotate 45°, keep the inscribed square (`rotate_and_crop`).
3. Run a PCA over all shots at this hold time and project out **PC1**. With
   no mean subtraction, PC1 is the shared box envelope. What is left is the
   domain pattern.

In [ ]:
DEMO_SET = "sep11"
DEMO_T = 200.0      # ms; nearest available hold time is used
SHOT = 0            # which repeat at that time

d = data[DEMO_SET]
ds = d["ds"]
PX = ds.pixel_um

t_demo = d["times"][np.argmin(np.abs(d["times"] - DEMO_T))]
idx_t = np.flatnonzero(d["t"] == t_demo)
_, pcs_demo = pca_svd(d["stack"][idx_t])

raw_full = ShotFile(d["paths"][idx_t[SHOT]], cam=ds.camera).od
od_crop = d["stack"][idx_t[SHOT]]
field, envelope = remove_components(od_crop, pcs_demo, 1)
ny, nx = field.shape
print(f"t = {t_demo:g} ms, shot {SHOT + 1}/{len(idx_t)}, field {ny}x{nx} px "
      f"= {ny * PX:.0f}x{nx * PX:.0f} um")

ext = [0, nx * PX, ny * PX, 0]     # image extent in um
r0, r1, c0, c1 = d["crop_box"]
fig, ax = plt.subplots(1, 4, figsize=(17, 4.2))
ax[0].imshow(raw_full, cmap="viridis", vmin=0, vmax=np.percentile(raw_full, 99.5))
ax[0].add_patch(Rectangle((c0, r0), c1 - c0, r1 - r0, fill=False, ec="r", lw=1.5))
ax[0].set_title("raw OD, full frame")
ax[1].imshow(od_crop, cmap="viridis", extent=ext)
ax[1].set_title("crop + rotate 45° + inscribed square")
ax[2].imshow(envelope, cmap="viridis", extent=ext)
ax[2].set_title("PC1 projection (removed)")
lim = np.percentile(np.abs(field), 99.5)
ax[3].imshow(field, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
ax[3].set_title("field = OD − PC1 part")
for a in ax[1:]:
    a.set_xlabel(r"x ($\mu$m)")
fig.tight_layout()
plt.show()

## Approach A: first moment of the structure factor

$$L_A = \frac{2\pi}{\langle k\rangle}, \qquad
\langle k\rangle = \frac{\sum_{\mathbf k} |\mathbf k|\,S(\mathbf k)}{\sum_{\mathbf k} S(\mathbf k)},
\qquad S(\mathbf k) = |\mathrm{FFT}[\delta n\, w]|^2,\; S(0)=0$$

### A1–A2: subtract the mean, then apply the Tukey window
Subtracting the mean means $S$ measures fluctuations rather than overall
brightness. The FFT treats the frame as periodic, so the left edge is glued to
the right edge. Any mismatch there becomes a sharp step that leaks power to
high $k$ and biases $L$ downward. The Tukey window ($\alpha$ = 0.5, the project
default) tapers only the outer border to zero and leaves the interior alone.

In [ ]:
dn = field - field.mean()                                   # A1
w = window_2d(field.shape, ds.calibration.window_alpha)     # A2
dnw = dn * w

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
lim = np.percentile(np.abs(dn), 99.5)
ax[0].imshow(dn, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
ax[0].set_title(r"A1: $\delta n$ = field − mean")
ax[1].imshow(w, cmap="gray", vmin=0, vmax=1, extent=ext)
ax[1].set_title(rf"A2: Tukey window, $\alpha$ = {ds.calibration.window_alpha}")
ax[2].imshow(dnw, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
ax[2].set_title(r"A2: $\delta n \cdot w$")
for a in ax:
    a.set_xlabel(r"x ($\mu$m)")
fig.tight_layout()
plt.show()

### A3–A4: structure factor $S(\mathbf k)$ and the $|\mathbf k|$ grid
$S$ is shown with $\mathbf k = 0$ at the centre (`fftshift`) on a log colour
scale. The uniform mode is set to zero. A domain pattern with no preferred
direction shows up as a ring, and the ring's radius sets the domain size.

In [ ]:
S = np.abs(np.fft.fft2(dnw)) ** 2                           # A3
S[0, 0] = 0.0

ky = 2 * np.pi * np.fft.fftfreq(ny)                         # A4, rad/px
kx = 2 * np.pi * np.fft.fftfreq(nx)
KX, KY = np.meshgrid(kx, ky, indexing="xy")
K = np.hypot(KX, KY)

k_um = lambda k: k / PX                                     # rad/px -> rad/um
kext = [k_um(np.fft.fftshift(kx)[0]), k_um(np.fft.fftshift(kx)[-1]),
        k_um(np.fft.fftshift(ky)[-1]), k_um(np.fft.fftshift(ky)[0])]
Ss = np.fft.fftshift(S)
K_ZOOM = 1.0  # rad/um, for the zoomed panel

fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
im = ax[0].imshow(np.log10(Ss + Ss[Ss > 0].min()), cmap="magma", extent=kext)
ax[0].set_title(r"A3: $\log_{10} S(\mathbf{k})$, full")
fig.colorbar(im, ax=ax[0], shrink=0.8)
im = ax[1].imshow(np.log10(Ss + Ss[Ss > 0].min()), cmap="magma", extent=kext)
ax[1].set_xlim(-K_ZOOM, K_ZOOM); ax[1].set_ylim(-K_ZOOM, K_ZOOM)
ax[1].set_title(r"A3: zoom on low $k$")
fig.colorbar(im, ax=ax[1], shrink=0.8)
im = ax[2].imshow(k_um(np.fft.fftshift(K)), cmap="cividis", extent=kext)
ax[2].set_title(r"A4: $|\mathbf{k}|$ (rad/$\mu$m)")
fig.colorbar(im, ax=ax[2], shrink=0.8)
for a in ax:
    a.set_xlabel(r"$k_x$ (rad/$\mu$m)")
    a.set_ylabel(r"$k_y$ (rad/$\mu$m)")
fig.tight_layout()
plt.show()

### A5: weight by $|\mathbf k|$, take the mean, and convert to $L_A$
$|\mathbf k|\,S$ is the integrand of the numerator. Dividing its sum by the sum
of $S$ gives the power-weighted mean wavevector $\langle k\rangle$, and
$L_A = 2\pi/\langle k\rangle$ is the wavelength that goes with it. The right
panel averages $S$ over angle so $\langle k\rangle$ can be seen against the
spectrum. The unwindowed value is printed alongside for comparison: the
700p notebook used no window.

In [ ]:
kS = K * S                                                   # A5
k_mean = kS.sum() / S.sum()
L_A_px = 2 * np.pi / k_mean
L_A = L_A_px * PX

# cross-check against the library estimator
assert np.isclose(L_A_px, domain_length_px(field, window=w))
L_A_nowin = domain_length_px(field) * PX

# angular average, for display only
dk = 2 * np.pi / min(ny, nx)
kbin = np.round(K / dk).astype(int)
cnt = np.bincount(kbin.ravel())
S_q = np.bincount(kbin.ravel(), S.ravel()) / np.maximum(cnt, 1)
q = k_um(np.arange(len(S_q)) * dk)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
im = ax[0].imshow(np.fft.fftshift(kS), cmap="magma", extent=kext)
ax[0].set_xlim(-K_ZOOM, K_ZOOM); ax[0].set_ylim(-K_ZOOM, K_ZOOM)
ax[0].add_patch(Circle((0, 0), k_um(k_mean), fill=False, ec="c", lw=1.5, ls="--"))
ax[0].set_title(r"A5: $|\mathbf{k}|\,S(\mathbf{k})$  (cyan: $|\mathbf{k}|=\langle k\rangle$)")
ax[0].set_xlabel(r"$k_x$ (rad/$\mu$m)")
fig.colorbar(im, ax=ax[0], shrink=0.8)

ax[1].semilogy(q[1:], S_q[1:], color="C0", label=r"$S(q)$")
ax[1].axvline(k_um(k_mean), color="c", ls="--",
              label=rf"$\langle k\rangle$ = {k_um(k_mean):.3f} rad/$\mu$m")
ax[1].set_xlabel(r"$q$ (rad/$\mu$m)")
ax[1].set_title("angle-averaged structure factor")
ax[1].legend(frameon=False)
ax[1].grid(alpha=0.25, which="both")

lim = np.percentile(np.abs(field), 99.5)
ax[2].imshow(field, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
y_bar = ny * PX * 0.92
ax[2].plot([5, 5 + L_A], [y_bar, y_bar], color="k", lw=4, solid_capstyle="butt")
ax[2].text(5, y_bar - 3, rf"$L_A$ = {L_A:.2f} $\mu$m", fontsize=11,
           bbox=dict(fc="w", ec="none", alpha=0.8))
ax[2].set_title("field with $L_A$ as a scale bar")
ax[2].set_xlabel(r"x ($\mu$m)")
fig.tight_layout()
plt.show()

print(f"<k> = {k_um(k_mean):.4f} rad/um")
print(f"L_A (Tukey window) = {L_A:.2f} um   [= domain_length_px(field, window=w) * px]")
print(f"L_A (no window)    = {L_A_nowin:.2f} um")

## Approach B: first zero of the pair correlation $g(r)$

$$C(\boldsymbol\Delta) = \frac{\sum_{\mathbf r}\delta n(\mathbf r)\,\delta n(\mathbf r+\boldsymbol\Delta)}{N(\boldsymbol\Delta)},
\qquad g(r) = \big\langle C(\boldsymbol\Delta)\big\rangle_{|\boldsymbol\Delta| = r} \big/ C(0),
\qquad L_B:\; g(L_B) = 0 \text{ (first crossing)}$$

$C$ is computed with FFTs (Wiener–Khinchin), but unlike approach A the frame
is **zero-padded**, so it is never treated as periodic and no window is needed.

### B1–B2: subtract the mean, then zero-pad to twice the size
Doubling the frame with zeros means the correlation at shift $\boldsymbol\Delta$
only multiplies real pixel pairs and never wraps around to the other side. The
mask records which pixels are real. Its own autocorrelation, computed in B4,
counts how many pairs exist at each shift.

In [ ]:
dn = field - field.mean()                                   # B1
padded = np.zeros((2 * ny, 2 * nx))                         # B2
padded[:ny, :nx] = dn
mask = np.zeros_like(padded)
mask[:ny, :nx] = 1.0

ext2 = [0, 2 * nx * PX, 2 * ny * PX, 0]
lim = np.percentile(np.abs(dn), 99.5)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.4))
ax[0].imshow(dn, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
ax[0].set_title(r"B1: $\delta n$ = field − mean")
ax[1].imshow(padded, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext2)
ax[1].set_title(r"B2: zero-padded to $2N \times 2N$")
ax[2].imshow(mask, cmap="gray", extent=ext2)
ax[2].set_title("B2: support mask")
for a in ax:
    a.set_xlabel(r"x ($\mu$m)")
fig.tight_layout()
plt.show()

### B3–B5: raw autocorrelation, pair counts, normalised $C(\boldsymbol\Delta)$
* **B3**: $\mathrm{IFFT}\,|\mathrm{FFT}(\text{padded})|^2$, the sum of
  $\delta n\,\delta n$ products at each shift. Large shifts have few
  overlapping pixels, so this sum shrinks toward the edges even with no
  physics behind it.
* **B4**: the same operation on the mask gives $N(\boldsymbol\Delta)$, the
  number of overlapping pixel pairs. It is a pyramid that peaks at
  $N_x N_y$ for zero shift.
* **B5**: dividing B3 by B4 gives an unbiased mean product at every shift.
  Normalising so $C(0)=1$ makes the black contour the $C=0$ line, where the
  field stops being correlated.

In [ ]:
C_raw = np.fft.fftshift(np.fft.ifft2(np.abs(np.fft.fft2(padded)) ** 2).real)   # B3
N_ov = np.fft.fftshift(np.fft.ifft2(np.abs(np.fft.fft2(mask)) ** 2).real)      # B4
C = C_raw / np.maximum(N_ov, 1.0)                                              # B5
C = C / C[ny, nx]                       # zero shift sits at (ny, nx) after fftshift

dext = [-nx * PX, nx * PX, ny * PX, -ny * PX]   # shift axes in um
D_ZOOM = 40.0  # um, half-width of the zoomed panel

fig, ax = plt.subplots(1, 4, figsize=(20, 4.4))
lim = np.abs(C_raw).max()
im = ax[0].imshow(C_raw, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=dext)
ax[0].set_title(r"B3: raw autocorrelation $\sum \delta n\,\delta n$")
fig.colorbar(im, ax=ax[0], shrink=0.8)
im = ax[1].imshow(N_ov, cmap="cividis", extent=dext)
ax[1].set_title(r"B4: overlap count $N(\Delta)$")
fig.colorbar(im, ax=ax[1], shrink=0.8)
im = ax[2].imshow(C, cmap="RdBu_r", vmin=-1, vmax=1, extent=dext)
ax[2].set_title(r"B5: $C(\Delta) = $ B3 / B4, $C(0)=1$")
fig.colorbar(im, ax=ax[2], shrink=0.8)
im = ax[3].imshow(C, cmap="RdBu_r", vmin=-0.3, vmax=0.3, extent=dext)
dy_um = (np.arange(2 * ny) - ny) * PX
dx_um = (np.arange(2 * nx) - nx) * PX
ax[3].contour(dx_um, dy_um, C, levels=[0], colors="k", linewidths=0.8)
ax[3].set_xlim(-D_ZOOM, D_ZOOM); ax[3].set_ylim(D_ZOOM, -D_ZOOM)
ax[3].set_title(r"B5 zoom: black = $C(\Delta)=0$")
fig.colorbar(im, ax=ax[3], shrink=0.8)
for a in ax:
    a.set_xlabel(r"$\Delta x$ ($\mu$m)")
    a.set_ylabel(r"$\Delta y$ ($\mu$m)")
fig.tight_layout()
plt.show()

### B6–B7: average over angle to get $g(r)$, then find its first zero
Every shift $\boldsymbol\Delta$ is binned by its length $r=|\boldsymbol\Delta|$
into 1 px rings, and each ring is averaged. The search stops at
$r_{max}=\min(N_x,N_y)-1$, beyond which the rings are incomplete. $L_B$ is the
first point where $g$ goes from positive to non-positive, with linear
interpolation between the two bracketing bins.

In [ ]:
yy, xx = np.indices(C.shape)                                  # B6
R_px = np.hypot(yy - ny, xx - nx)
r_max = min(ny, nx) - 1
sel = R_px <= r_max
ring = R_px.astype(int)[sel]
g = np.bincount(ring, C[sel], minlength=r_max + 1) / np.bincount(ring, minlength=r_max + 1)
g = g / g[0]
r_um = np.arange(r_max + 1) * PX

L_B = first_zero(r_um, g)                                     # B7

# cross-check against the library estimator
r_lib, g_lib = radial_autocorr(field, PX)
assert np.allclose(g, g_lib) and np.isclose(L_B, first_zero(r_lib, g_lib))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
im = ax[0].imshow(np.where(sel, R_px, np.nan) * PX, cmap="cividis", extent=dext)
ax[0].set_title(r"B6: ring index $r=|\Delta|$ ($\mu$m), $r \leq r_{max}$")
ax[0].set_xlabel(r"$\Delta x$ ($\mu$m)")
fig.colorbar(im, ax=ax[0], shrink=0.8)

rng = np.random.default_rng(0)
pick = rng.choice(sel.sum(), size=min(20000, sel.sum()), replace=False)
ax[1].plot((R_px[sel] * PX)[pick], C[sel][pick], ",", color="0.6", alpha=0.5,
           label=r"individual $C(\Delta)$")
ax[1].plot(r_um, g, color="C0", lw=2, label=r"$g(r)$ (ring average)")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].axvline(L_B, color="C3", ls="--", label=rf"first zero $L_B$ = {L_B:.2f} $\mu$m")
ax[1].set_xlim(0, min(4 * L_B, r_um[-1]) if np.isfinite(L_B) else r_um[-1])
ax[1].set_ylim(-0.5, 1.05)
ax[1].set_xlabel(r"$r$ ($\mu$m)")
ax[1].set_title("B7: pair correlation and its first zero")
ax[1].legend(frameon=False)
ax[1].grid(alpha=0.25)

lim = np.percentile(np.abs(field), 99.5)
ax[2].imshow(field, cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
for cx, cy in [(0.3, 0.3), (0.7, 0.3), (0.3, 0.7), (0.7, 0.7)]:
    ax[2].add_patch(Circle((cx * nx * PX, cy * ny * PX), L_B, fill=False, ec="k", lw=1.2))
ax[2].set_title(rf"field with circles of radius $L_B$ = {L_B:.2f} $\mu$m")
ax[2].set_xlabel(r"x ($\mu$m)")
fig.tight_layout()
plt.show()

print(f"L_B (first zero of g) = {L_B:.2f} um   [= first_zero(*radial_autocorr(field, px))]")

### The two numbers side by side

$L_A$ and $L_B$ are different definitions of length, and their ratio is not
fixed. Take an idealised isotropic pattern with a single wavevector $k_0$.
Then $L_A = 2\pi/k_0$, one full domain period. Since $g(r) = J_0(k_0 r)$, whose
first zero is at $2.405/k_0$, it also gives $L_B \approx 0.38\,L_A$.

Real shots do not behave like that. The angle-averaged $S(q)$ in step A5 has a
flat shot-noise plateau out to about 5 rad/$\mu$m. That plateau carries real
weight in $\langle k\rangle$, which pulls $\langle k\rangle$ up and $L_A$
down. Approach A also depends on the window: compare the windowed and
unwindowed values below. In $g(r)$, uncorrelated noise only adds a spike at
$r = 0$ and leaves the zero crossing where it was. That is why
`scripts/analysis/coarsening_scaling_analysis.py` uses approach B for $L(t)$.
Whichever estimator is used, compare its value only with itself across hold
times.

In [ ]:
print(f"dataset {DEMO_SET}, t = {t_demo:g} ms, shot {SHOT + 1}")
print(f"  A  structure-factor moment, Tukey window : L_A = {L_A:6.2f} um")
print(f"  A  structure-factor moment, no window    : L_A = {L_A_nowin:6.2f} um")
print(f"  B  first zero of g(r)                    : L_B = {L_B:6.2f} um")
print(f"  L_B / L_A = {L_B / L_A:.2f}")

## L(t) for the 2026-09-11 data: both approaches

The step-by-step chain above is now run on **every** 09-11 shot. Each shot has
PC1 of its own hold time's PCA projected out, and then three estimates are
computed:

* **A, Tukey window**: `domain_length_px(field, window=w)`, the project default
* **A, no window**: `domain_length_px(field)`, as used in `pca_data_images.ipynb`
* **B**: `first_zero(*radial_autocorr(field, px))`

Points are the mean over repeats ± SEM.
If $g(r)$ never crosses zero inside $r_{max}$, the shot's $L_B$ is NaN. Those
shots are left out of that time's mean and the number dropped is printed.

In [ ]:
d11 = data["sep11"]
px11 = d11["ds"].pixel_um
w11 = window_2d(d11["stack"].shape[1:], d11["ds"].calibration.window_alpha)
t11 = d11["t"]
times11 = d11["times"]

L_A_win = np.empty(len(t11))
L_A_now = np.empty(len(t11))
L_B_all = np.empty(len(t11))
for t in times11:
    idx = np.flatnonzero(t11 == t)
    _, pcs = pca_svd(d11["stack"][idx])          # this hold time's own basis
    for i in idx:
        f, _ = remove_components(d11["stack"][i], pcs, 1)
        L_A_win[i] = domain_length_px(f, window=w11) * px11
        L_A_now[i] = domain_length_px(f) * px11
        L_B_all[i] = first_zero(*radial_autocorr(f, px11))


def per_time(L):
    # mean, SEM and number of finite values at each hold time
    out = []
    for t in times11:
        v = L[t11 == t]
        v = v[np.isfinite(v)]
        out.append((v.mean(), v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan, len(v)))
    return [np.array(c) for c in zip(*out)]


series = {
    "A: structure-factor moment, Tukey window": (L_A_win, *per_time(L_A_win)),
    "A: structure-factor moment, no window": (L_A_now, *per_time(L_A_now)),
    "B: first zero of g(r)": (L_B_all, *per_time(L_B_all)),
}

n_tot = np.array([(t11 == t).sum() for t in times11])
print(f"{'t (ms)':>8} {'n':>3} | {'L_A win':>14} | {'L_A no-win':>14} | {'L_B':>14} {'NaN':>4}")
for i, t in enumerate(times11):
    row = [f"{s[1][i]:6.2f} ± {s[2][i]:5.2f}" for s in series.values()]
    print(f"{t:8g} {n_tot[i]:3d} | {row[0]:>14} | {row[1]:>14} | {row[2]:>14} "
          f"{n_tot[i] - series['B: first zero of g(r)'][3][i]:4d}")

In [ ]:
# series to draw (the Tukey-window A is computed and tabulated above but not plotted)
styles = {
    "A: structure-factor moment, no window": dict(color="C0", marker="o"),
    "B: first zero of g(r)": dict(color="C3", marker="s"),
}


def plot_L_vs_t(series, title, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    for label in styles:
        _, mean, sem, _ = series[label]
        ax.errorbar(times11, mean, yerr=sem, ms=5, capsize=2, ls="-", lw=1,
                    label=label, **styles[label])
    ax.set_xlabel("hold time  t  (ms)")
    ax.set_ylabel(r"domain size  L  ($\mu$m)")
    ax.set_title(title)
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(d11["ds"].figure(fname), dpi=150)
    plt.show()


plot_L_vs_t(series, "2026-09-11: domain size vs hold time (PC1 removed)",
            "L_vs_t_both_approaches.png")

**Reading the plot**

* **20–150 ms:** all three estimates agree to within about 1–2 µm and grow
  from roughly 6–9 µm to about 16 µm. Over 20–40 ms, A rises faster than B
  (5.6 → 12 µm against 8.4 → 9.9 µm).
* **From about 250 ms on, the approaches split.** A levels off near 20 µm and
  then falls to about 15 µm by 2000 ms. B keeps growing to about 38–40 µm. A
  falling domain size is not physical. It matches the high-$k$ noise floor
  taking a larger share of $\langle k\rangle$ as contrast drops (step A5),
  and B is not affected by that noise. Treat B's late values with some care,
  though. At about 40 µm, L_B is a third of the 120 µm frame, so the zero
  crossing depends on only a few domains per shot.
* **The t = 0 value of L_B (about 31 µm) is not a domain size.** At t = 0
  there are no domains. After PC1 is removed, the field is shot noise plus
  whatever slow envelope residue PC1 missed. The noise only affects
  $g(0)$, so the zero crossing is set by the scale of that residue. Leave
  this point out of any fit.
* The Tukey window makes A about 1–2 µm larger at intermediate times, which is
  the downward bias the window is there to remove.

## Same L(t), with no PC removed

The same three estimators run directly on the cropped OD, with **no PC
removed**. With the no-window version of A, this reproduces the
"cropped OD only" plot from `pca_data_images.ipynb` (8.95 µm at 0 ms,
23.2 µm at 2000 ms).

**Why the two A curves differ at late times.** The estimator is identical.
The input is different.

* **Without PC removal**, the box envelope (walls and density profile) is part
  of the field. It carries 28–78% of the fluctuation power, depending on the
  hold time, and its own $L$ is long: about 24 µm early, rising to about 47 µm
  at 2000 ms. $L_A$ is a power-weighted blend, so as the domain contrast fades
  after about 300 ms, the envelope's share grows and pulls $L_A$ **up**. The
  raw curve keeps rising, but the envelope is responsible for part of that
  rise.
* **With PC1 removed**, the envelope is gone, but the shot-noise plateau
  remains. The cleaned-field rms falls from 0.196 at 300 ms to 0.136 at
  2000 ms. Over the same range, the share of $S$ beyond 1.5 rad/µm rises from
  5% to 9%, and that pulls $L_A$ **down**.

Neither late-time trend of A is a clean measurement of domain growth. Each one
is biased by whatever is left in the field. The envelope also sets a floor:
with no PC removed, L(t = 0) is 9 µm even though there are no domains yet.

In [ ]:
L_A_win_raw = np.empty(len(t11))
L_A_now_raw = np.empty(len(t11))
L_B_raw = np.empty(len(t11))
for i, od in enumerate(d11["stack"]):          # cropped OD, nothing removed
    L_A_win_raw[i] = domain_length_px(od, window=w11) * px11
    L_A_now_raw[i] = domain_length_px(od) * px11
    L_B_raw[i] = first_zero(*radial_autocorr(od, px11))

series_raw = {
    "A: structure-factor moment, Tukey window": (L_A_win_raw, *per_time(L_A_win_raw)),
    "A: structure-factor moment, no window": (L_A_now_raw, *per_time(L_A_now_raw)),
    "B: first zero of g(r)": (L_B_raw, *per_time(L_B_raw)),
}

print(f"{'t (ms)':>8} | {'L_A win':>14} | {'L_A no-win':>14} | {'L_B':>14} {'NaN':>4}")
for i, t in enumerate(times11):
    row = [f"{s[1][i]:6.2f} ± {s[2][i]:5.2f}" for s in series_raw.values()]
    print(f"{t:8g} | {row[0]:>14} | {row[1]:>14} | {row[2]:>14} "
          f"{n_tot[i] - series_raw['B: first zero of g(r)'][3][i]:4d}")

plot_L_vs_t(series_raw, "2026-09-11: domain size vs hold time (no PC removed)",
            "L_vs_t_both_approaches_noPC.png")


# Part 2: first four principal components vs hold time

Each hold time gets **its own** PCA, using `pca_svd` on that time's shots
without subtracting the mean, so PC1 is the shared envelope. A movie frame
shows PC1 through PC4 for one hold time, and the panel titles give each
component's share of the total variance.

An SVD fixes each mode only up to an overall sign, so a mode could flip
colour at random between frames. `align_signs` removes that by flipping each
frame to correlate positively with the previous one. Every component keeps one
fixed symmetric colour scale across all frames. The eigenvectors have unit
norm, so a change in contrast between frames is real.

Each movie is also saved as a GIF to `results/figures/<dataset>/`.

In [ ]:
N_PC = 4


def pc_movie(name, n_pc=N_PC, fps=1):
    d = data[name]
    times = d["times"]
    pca = {t: pca_svd(d["stack"][d["t"] == t]) for t in times}
    n_shots = {t: int((d["t"] == t).sum()) for t in times}

    # frames[k][i] = PC(k+1) at times[i], sign-aligned along time
    frames = [align_signs([pca[t][1][..., k] for t in times]) for k in range(n_pc)]
    lims = [np.percentile(np.abs(np.stack(fr)), 99.5) for fr in frames]

    fig, axes = plt.subplots(1, n_pc, figsize=(3.6 * n_pc, 4.0))
    ims, titles = [], []
    for k, ax in enumerate(axes):
        ims.append(ax.imshow(frames[k][0], cmap="RdBu_r", vmin=-lims[k], vmax=lims[k]))
        titles.append(ax.set_title(""))
        ax.set_xticks([]); ax.set_yticks([])
    sup = fig.suptitle(" ")
    fig.tight_layout(rect=(0, 0, 1, 0.92))  # leave room for the suptitle

    def update(i):
        t = times[i]
        evals = pca[t][0]
        for k in range(n_pc):
            ims[k].set_data(frames[k][i])
            titles[k].set_text(f"PC{k + 1}  ({evals[k]:.2%} of variance)")
        sup.set_text(f"{name}: t = {t:g} ms   ({n_shots[t]} shots)")
        return ims + titles + [sup]

    anim = animation.FuncAnimation(fig, update, frames=len(times), interval=1000 / fps)
    gif = d["ds"].figure(f"pc1-{n_pc}_vs_time.gif")
    anim.save(gif, writer=animation.PillowWriter(fps=fps))
    print(f"saved {gif}")
    plt.close(fig)
    return HTML(anim.to_jshtml())

## 700p shots (2026-08-19)

In [ ]:
display(pc_movie("shots700p"))

## 2026-09-11 shots

In [ ]:
display(pc_movie("sep11"))